In [17]:
from pathlib import Path

import pandas as pd
import numpy as np

In [18]:
DATA_DIR = Path("../data/processed")

solexs = pd.read_parquet(
    DATA_DIR / "solexs_flare_catalog.parquet"
)

hel1os = pd.read_parquet(
    DATA_DIR / "aligned_dataset.parquet"
)

print(solexs.shape)
print(hel1os.shape)

(11, 8)
(43188, 151)


In [19]:
solexs["peak_time"] = pd.to_datetime(
    solexs["peak_time"]
).astype("datetime64[ns]")

hel1os["DATETIME"] = pd.to_datetime(
    hel1os["DATETIME"]
).astype("datetime64[ns]")

In [20]:
master = pd.merge_asof(
    solexs.sort_values("peak_time"),
    hel1os.sort_values("DATETIME"),
    left_on="peak_time",
    right_on="DATETIME",
    direction="nearest",
    tolerance=pd.Timedelta("5min"),
)

In [21]:
print(master.shape)

master.head()

(11, 159)


,start_time,peak_time,end_time,peak_counts,duration_sec,integrated_counts_x,confidence,severity,MJD,ISOT,...,sun2yawdeg_gradient,sun2rolldeg_gradient,sun2pitchdeg_gradient,czt_balance,cdte_balance,total_detector_counts,czt1ctr_rolling_mean,czt2ctr_rolling_mean,cdte1ctr_rolling_mean,cdte2ctr_rolling_mean
0,2026-06-24 01:02:57,2026-06-24 01:03:27,2026-06-24 01:05:10,124.0,133.0,15460.0,0.790,High,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-06-24 03:52:31,2026-06-24 03:52:34,2026-06-24 03:53:07,109.0,36.0,3493.0,0.455,Moderate,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-06-24 06:50:12,2026-06-24 06:51:21,2026-06-24 06:51:44,81.0,92.0,8194.0,0.471,Moderate,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-06-24 07:09:50,2026-06-24 07:10:31,2026-06-24 07:12:00,163.0,130.0,16826.0,0.956,High,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-06-24 09:02:05,2026-06-24 09:03:03,2026-06-24 09:04:12,58.0,127.0,5755.0,0.335,Low,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
missing = master.isna().sum()

missing[missing > 0].sort_values()

MJD                      6
ISOT                     6
COUNTS                   6
STAT_ERR                 6
DATETIME                 6
                        ..
total_detector_counts    6
czt1ctr_rolling_mean     6
czt2ctr_rolling_mean     6
cdte1ctr_rolling_mean    6
cdte2ctr_rolling_mean    6
Length: 151, dtype: int64

In [23]:
master.to_csv(
    DATA_DIR / "master_flare_catalog.csv",
    index=False,
)

master.to_parquet(
    DATA_DIR / "master_flare_catalog.parquet",
    index=False,
)

In [24]:
print(master.shape)
print(master.isna().sum().sum())

(11, 159)
906


In [25]:
missing = (
    master.isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0]

rolling_std              6
DATETIME                 6
rolling_mean             6
STAT_ERR                 6
COUNTS                   6
                        ..
total_detector_counts    6
czt1ctr_rolling_mean     6
czt2ctr_rolling_mean     6
cdte1ctr_rolling_mean    6
cdte2ctr_rolling_mean    6
Length: 151, dtype: int64

In [26]:
master.columns[master.isna().any()].tolist()

['MJD',
 'ISOT',
 'COUNTS',
 'STAT_ERR',
 'DATETIME',
 'rolling_mean',
 'rolling_std',
 'rolling_max',
 'rolling_min',
 'ema_10',
 'ema_30',
 'diff1',
 'diff2',
 'gradient',
 'lag_1',
 'lag_2',
 'lag_5',
 'lag_10',
 'window_energy',
 'window_variance',
 'rolling_skew',
 'rolling_kurtosis',
 'rolling_rms',
 'rolling_median',
 'rolling_mad',
 'rolling_cv',
 'z_score',
 'robust_z',
 'peak_to_peak',
 'local_energy',
 'is_peak',
 'peak_prominence',
 'peak_height',
 'peak_width',
 'time_since_last_peak',
 'rise_rate',
 'decay_rate',
 'peak_density',
 'integrated_counts_y',
 'peak_rank',
 'strong_peak',
 'flare_candidate',
 'SPEC_NUM',
 'ROWID',
 'TSTART',
 'TSTOP',
 'EXPOSURE',
 'MID_TIME',
 'spec_total_counts',
 'spec_mean_counts',
 'spec_std_counts',
 'spec_max',
 'spec_min',
 'peak_channel',
 'spectral_centroid',
 'spectral_spread',
 'spectral_entropy',
 'low_energy',
 'mid_energy',
 'high_energy',
 'hardness_ratio1',
 'hardness_ratio2',
 'dominant_fraction',
 'active_channels',
 'l0recnu

In [27]:
master[
    master["DATETIME"].isna()
][
    ["peak_time", "severity", "peak_counts"]
]

,peak_time,severity,peak_counts
0,2026-06-24 01:03:27,High,124.0
1,2026-06-24 03:52:34,Moderate,109.0
2,2026-06-24 06:51:21,Moderate,81.0
3,2026-06-24 07:10:31,High,163.0
4,2026-06-24 09:03:03,Low,58.0
5,2026-06-24 10:03:47,Low,49.0


In [28]:
print(hel1os["DATETIME"].min())
print(hel1os["DATETIME"].max())

print(solexs["peak_time"].min())
print(solexs["peak_time"].max())

2026-06-24 12:00:01.739000
2026-06-24 23:59:48.739000
2026-06-24 01:03:27
2026-06-24 22:17:54


In [29]:
master = master.dropna(subset=["DATETIME"]).reset_index(drop=True)

print(master.shape)

(5, 159)
